# Week 1: Data Model, Data Cleaning & Preprocessing
# Project: Supply Chain Dataset Analysis


In [8]:
from google.colab import files
uploaded = files.upload()


Saving supply_chain_data.csv to supply_chain_data (1).csv


In [9]:
import pandas as pd
import numpy as np

**1. Load the dataset**


In [10]:
df = pd.read_csv('/content/supply_chain_data.csv')

print("Shape of dataset:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Shape of dataset: (100, 24)

Column names:
['Product type', 'SKU', 'Price', 'Availability', 'Number of products sold', 'Revenue generated', 'Customer demographics', 'Stock levels', 'Lead times', 'Order quantities', 'Shipping times', 'Shipping carriers', 'Shipping costs', 'Supplier name', 'Location', 'Lead time', 'Production volumes', 'Manufacturing lead time', 'Manufacturing costs', 'Inspection results', 'Defect rates', 'Transportation modes', 'Routes', 'Costs']

Data types:
Product type                object
SKU                         object
Price                      float64
Availability                 int64
Number of products sold      int64
Revenue generated          float64
Customer demographics       object
Stock levels                 int64
Lead times                   int64
Order quantities             int64
Shipping times               int64
Shipping carriers           object
Shipping costs             float64
Supplier name               object
Location                    ob

**2. Initial inspection**

In [11]:
print("\nFirst 5 rows:")
print(df.head())

print("\nSummary statistics:")
print(df.describe())



First 5 rows:
  Product type   SKU      Price  Availability  Number of products sold  \
0     haircare  SKU0  69.808006            55                      802   
1     skincare  SKU1  14.843523            95                      736   
2     haircare  SKU2  11.319683            34                        8   
3     skincare  SKU3  61.163343            68                       83   
4     skincare  SKU4   4.805496            26                      871   

   Revenue generated Customer demographics  Stock levels  Lead times  \
0        8661.996792            Non-binary            58           7   
1        7460.900065                Female            53          30   
2        9577.749626               Unknown             1          10   
3        7766.836426            Non-binary            23          13   
4        2686.505152            Non-binary             5           3   

   Order quantities  ...  Location Lead time  Production volumes  \
0                96  ...    Mumbai     

 3. Data Cleaning

3.1 Check for missing values


In [12]:
print("\nMissing values per column:")
print(df.isnull().sum())


Missing values per column:
Product type               0
SKU                        0
Price                      0
Availability               0
Number of products sold    0
Revenue generated          0
Customer demographics      0
Stock levels               0
Lead times                 0
Order quantities           0
Shipping times             0
Shipping carriers          0
Shipping costs             0
Supplier name              0
Location                   0
Lead time                  0
Production volumes         0
Manufacturing lead time    0
Manufacturing costs        0
Inspection results         0
Defect rates               0
Transportation modes       0
Routes                     0
Costs                      0
dtype: int64


3.2 Check for duplicate rows

In [13]:
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate SKUs:", df['SKU'].duplicated().sum())


Duplicate rows: 0
Duplicate SKUs: 0


3.3 Check for invalid / out-of-range values

In [14]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    n_negative = (df[col] < 0).sum()
    if n_negative > 0:
        print(f"WARNING: {col} has {n_negative} negative values")
print("\nNo negative values found in numeric columns.")


No negative values found in numeric columns.


3.4 Standardize text/categorical columns (strip whitespace, fix casing)

In [15]:
categorical_cols = df.select_dtypes(include=['object', 'string']).columns
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()

print("\nUnique values for key categorical columns:")
for col in ['Product type', 'Customer demographics', 'Shipping carriers',
            'Supplier name', 'Location', 'Inspection results',
            'Transportation modes', 'Routes']:
    print(f"  {col}: {df[col].unique().tolist()}")


Unique values for key categorical columns:
  Product type: ['haircare', 'skincare', 'cosmetics']
  Customer demographics: ['Non-binary', 'Female', 'Unknown', 'Male']
  Shipping carriers: ['Carrier B', 'Carrier A', 'Carrier C']
  Supplier name: ['Supplier 3', 'Supplier 1', 'Supplier 5', 'Supplier 4', 'Supplier 2']
  Location: ['Mumbai', 'Kolkata', 'Delhi', 'Bangalore', 'Chennai']
  Inspection results: ['Pending', 'Fail', 'Pass']
  Transportation modes: ['Road', 'Air', 'Rail', 'Sea']
  Routes: ['Route B', 'Route C', 'Route A']


3.5 Outlier check using IQR method

In [16]:
def iqr_outlier_report(series, name):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = series[(series < lower) | (series > upper)]
    print(f"  {name}: {len(outliers)} potential outliers "
          f"(range checked: {lower:.2f} to {upper:.2f})")

print("\nOutlier scan (IQR method):")
for col in ['Price', 'Revenue generated', 'Shipping costs',
            'Manufacturing costs', 'Defect rates', 'Costs']:
    iqr_outlier_report(df[col], col)


Outlier scan (IQR method):
  Price: 0 potential outliers (range checked: -66.80 to 163.60)
  Revenue generated: 0 potential outliers (range checked: -5348.85 to 16415.67)
  Shipping costs: 0 potential outliers (range checked: -2.55 to 13.69)
  Manufacturing costs: 0 potential outliers (range checked: -45.47 to 137.08)
  Defect rates: 0 potential outliers (range checked: -2.82 to 7.40)
  Costs: 0 potential outliers (range checked: -347.67 to 1429.53)


4. Build a simple data model

 Profit margin per unit sold = Revenue - (Manufacturing cost * units) - shipping/transport cost

In [17]:
df['Total manufacturing cost'] = df['Manufacturing costs'] * df['Production volumes']
df['Profit'] = df['Revenue generated'] - df['Costs']
df['Profit margin %'] = (df['Profit'] / df['Revenue generated']) * 100

 Revenue per unit sold


In [18]:
df['Revenue per unit'] = df['Revenue generated'] / df['Number of products sold']

Defect flag

In [19]:
df['Has defect issue'] = df['Defect rates'] > df['Defect rates'].median()

print("\nNew engineered columns added:")
print(df[['Profit', 'Profit margin %', 'Revenue per unit', 'Has defect issue']].head())


New engineered columns added:
        Profit  Profit margin %  Revenue per unit  Has defect issue
0  8474.244717        97.832462         10.800495             False
1  6957.834486        93.257307         10.137092              True
2  9435.829344        98.518229       1197.218703              True
3  7512.060266        96.719692         93.576342              True
4  1763.064520        65.626694          3.084392              True


**5. Save cleaned dataset**

In [20]:
df.to_csv('supply_chain_data_cleaned.csv', index=False)
df.to_excel('supply_chain_data_cleaned.xlsx', index=False)
print("\nCleaned dataset saved as 'supply_chain_data_cleaned.csv' and '.xlsx'")
print("Final shape:", df.shape)


Cleaned dataset saved as 'supply_chain_data_cleaned.csv' and '.xlsx'
Final shape: (100, 29)


In [21]:
import os
print(os.listdir('/content'))

['.config', 'supply_chain_data (1).csv', 'supply_chain_data_cleaned.csv', 'supply_chain_data.csv', 'supply_chain_data_cleaned.xlsx', 'sample_data']


In [23]:
from google.colab import files
files.download('/content/supply_chain_data_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:

import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('/content/supply_chain_data_cleaned.csv')
plt.rcParams['figure.figsize'] = (8, 5)


Q1: What is the impact of product type on revenue?


In [26]:
q1 = df.groupby('Product type')['Revenue generated'].agg(['sum', 'mean', 'count'])
print("Q1: Revenue by product type")
print(q1)

plt.figure()
sns.barplot(data=df, x='Product type', y='Revenue generated', estimator=sum, errorbar=None)
plt.title('Total Revenue by Product Type')
plt.ylabel('Total Revenue')
plt.tight_layout()
plt.savefig('q1_revenue_by_product_type.png', dpi=120)
plt.close()

Q1: Revenue by product type
                        sum         mean  count
Product type                                   
cosmetics     161521.265999  6212.356385     26
haircare      174455.390605  5131.040900     34
skincare      241628.162133  6040.704053     40


Q2: Which supplier contributes most to defect rates?

In [27]:
q2 = df.groupby('Supplier name')['Defect rates'].mean().sort_values(ascending=False)
print("\nQ2: Average defect rate by supplier")
print(q2)

plt.figure()
sns.barplot(x=q2.index, y=q2.values)
plt.title('Average Defect Rate by Supplier')
plt.ylabel('Average Defect Rate (%)')
plt.tight_layout()
plt.savefig('q2_defect_rate_by_supplier.png', dpi=120)
plt.close()



Q2: Average defect rate by supplier
Supplier name
Supplier 5    2.665408
Supplier 3    2.465786
Supplier 2    2.362750
Supplier 4    2.337397
Supplier 1    1.803630
Name: Defect rates, dtype: float64


Q3: How do shipping carriers compare in cost and shipping time?

In [28]:
q3 = df.groupby('Shipping carriers')[['Shipping costs', 'Shipping times']].mean()
print("\nQ3: Avg shipping cost & time by carrier")
print(q3)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
sns.barplot(data=df, x='Shipping carriers', y='Shipping costs', estimator='mean', errorbar=None, ax=axes[0])
axes[0].set_title('Avg Shipping Cost by Carrier')
sns.barplot(data=df, x='Shipping carriers', y='Shipping times', estimator='mean', errorbar=None, ax=axes[1])
axes[1].set_title('Avg Shipping Time by Carrier')
plt.tight_layout()
plt.savefig('q3_carrier_comparison.png', dpi=120)
plt.close()


Q3: Avg shipping cost & time by carrier
                   Shipping costs  Shipping times
Shipping carriers                                
Carrier A                5.554923        6.142857
Carrier B                5.509247        5.302326
Carrier C                5.599292        6.034483


Q4: Which transportation mode is the most cost-efficient?

In [29]:
q4 = df.groupby('Transportation modes')['Costs'].mean().sort_values()
print("\nQ4: Avg cost by transportation mode")
print(q4)

plt.figure()
sns.barplot(x=q4.index, y=q4.values)
plt.title('Average Total Cost by Transportation Mode')
plt.ylabel('Average Cost')
plt.tight_layout()
plt.savefig('q4_cost_by_transport_mode.png', dpi=120)
plt.close()



Q4: Avg cost by transportation mode
Transportation modes
Sea     417.819148
Rail    541.747556
Road    553.385988
Air     561.712596
Name: Costs, dtype: float64


Q5: Is there a relationship between price and number of products sold?

In [30]:
corr_price_sold = df['Price'].corr(df['Number of products sold'])
print(f"\nQ5: Correlation between Price and Units Sold: {corr_price_sold:.3f}")

plt.figure()
sns.scatterplot(data=df, x='Price', y='Number of products sold', hue='Product type')
plt.title('Price vs Number of Products Sold')
plt.tight_layout()
plt.savefig('q5_price_vs_units_sold.png', dpi=120)
plt.close()


Q5: Correlation between Price and Units Sold: 0.006


Q6: Which location (city) generates the highest profit margin?

In [31]:
q6 = df.groupby('Location')['Profit margin %'].mean().sort_values(ascending=False)
print("\nQ6: Avg profit margin by location")
print(q6)

plt.figure()
sns.barplot(x=q6.index, y=q6.values)
plt.title('Average Profit Margin % by Location')
plt.ylabel('Profit Margin %')
plt.tight_layout()
plt.savefig('q6_profit_margin_by_location.png', dpi=120)
plt.close()


Q6: Avg profit margin by location
Location
Mumbai       90.155022
Bangalore    87.036114
Kolkata      87.029201
Chennai      86.361085
Delhi        86.092901
Name: Profit margin %, dtype: float64


Q7: How does inspection result (Pass/Fail/Pending) relate to defect rate?

In [32]:
q7 = df.groupby('Inspection results')['Defect rates'].mean()
print("\nQ7: Avg defect rate by inspection result")
print(q7)

plt.figure()
sns.barplot(x=q7.index, y=q7.values)
plt.title('Average Defect Rate by Inspection Result')
plt.ylabel('Defect Rate (%)')
plt.tight_layout()
plt.savefig('q7_defect_by_inspection.png', dpi=120)
plt.close()

print("\nAll Week 2 charts saved as PNG files.")



Q7: Avg defect rate by inspection result
Inspection results
Fail       2.569302
Pass       2.039043
Pending    2.154218
Name: Defect rates, dtype: float64

All Week 2 charts saved as PNG files.
